In [1]:
import pandas as pd
from src.datasets.birdclef_waveform_dataset import BirdClefWaveformDataset, BirdClefWaveformDatasetArgs
from src.args.yaml_config import YamlConfig, YamlConfigModel
from src.models.perch_model import PerchModel
import torch
from src.train.evaluator import Evaluator
from tqdm import tqdm


yaml_config = YamlConfig().config

ds_args = BirdClefWaveformDatasetArgs(only_soundscapes=True)

ds = BirdClefWaveformDataset(ds_args, yaml_config).get_split("train")
model = PerchModel(yaml_config)

2026-05-05 19:16:48.379478: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Number of exact matches: 203
Number of missing mappings: 31


I0000 00:00:1778001414.897379 3118310 gpu_device.cc:2020] Created device /device:GPU:0 with 38485 MB memory:  -> device: 0, name: NVIDIA A100-PCIE-40GB, pci bus id: 0000:25:00.0, compute capability: 8.0
I0000 00:00:1778001415.146802 3118310 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38485 MB memory:  -> device: 0, name: NVIDIA A100-PCIE-40GB, pci bus id: 0000:25:00.0, compute capability: 8.0


In [2]:
dataloader_test = torch.utils.data.DataLoader(ds, batch_size=1, shuffle=True, collate_fn=ds.get_collate_fn())
evaluator = Evaluator(mode="test")

In [3]:
for batch in tqdm(dataloader_test):
    probabilities = model.forward(batch)
    loss = model.compute_loss(probabilities, batch)
    evaluator.track_batch(probabilities, loss, batch)
results = evaluator.evaluate()

  0%|          | 0/948 [00:00<?, ?it/s]

2026-05-05 19:16:59.654629: I external/local_xla/xla/service/service.cc:163] XLA service 0x7be72406dcb0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-05-05 19:16:59.654654: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA A100-PCIE-40GB, Compute Capability 8.0
2026-05-05 19:16:59.944954: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-05-05 19:16:59.982758: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002
2026-05-05 19:17:00.704980: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_55', 68 bytes spill stores, 68 bytes spill loads

2026-05-05 19:17:00.727848: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warn

In [4]:
results.get_average()

MetricEntry(metrics={'accuracy': 0.9769283490341927, 'bce_loss': 1.5163131856717138, 'f1': 0.26615933684105614, 'sensitivity': 0.26615933684105614, 'specificity': 0.9879653740532791}, loss=1.5163131856717138)